# Islamic Audiobook Generator - Backend
### Run each cell in order. Make sure Runtime > Change runtime type > T4 GPU is selected.

In [ ]:
# Step 1: Install dependencies
# Coqui TTS requires Python <3.12, install from GitHub for latest Colab compatibility
!pip install -q pdfplumber pydub fastapi uvicorn python-multipart pyngrok
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q git+https://github.com/coqui-ai/TTS.git

In [ ]:
# Step 2: Check GPU
!nvidia-smi

In [ ]:
# Step 3: Create directories
import os
os.makedirs('uploads', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
os.makedirs('voice_samples', exist_ok=True)
print('Directories created.')

In [ ]:
# Step 4: Write FastAPI app
app_code = '''
import os
import uuid
import pdfplumber
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from TTS.api import TTS
from pydub import AudioSegment

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")

def extract_text(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

def chunk_text(text, max_chars=200):
    sentences = text.replace("\\n", " ").split(". ")
    chunks, current = [], ""
    for s in sentences:
        if len(current) + len(s) < max_chars:
            current += s + ". "
        else:
            if current:
                chunks.append(current.strip())
            current = s + ". "
    if current:
        chunks.append(current.strip())
    return chunks

@app.post("/generate")
async def generate_audiobook(book: UploadFile = File(...), voice: UploadFile = File(...)):
    job_id = str(uuid.uuid4())
    book_path = f"uploads/{job_id}_book.pdf"
    voice_path = f"voice_samples/{job_id}_voice.wav"

    with open(book_path, "wb") as f:
        f.write(await book.read())
    with open(voice_path, "wb") as f:
        f.write(await voice.read())

    text = extract_text(book_path)
    if not text.strip():
        raise HTTPException(status_code=400, detail="Could not extract text from PDF")

    chunks = chunk_text(text)
    audio_segments = []

    for i, chunk in enumerate(chunks):
        chunk_path = f"outputs/{job_id}_chunk_{i}.wav"
        tts.tts_to_file(text=chunk, speaker_wav=voice_path, language="en", file_path=chunk_path)
        audio_segments.append(AudioSegment.from_wav(chunk_path))

    combined = sum(audio_segments)
    output_path = f"outputs/{job_id}_audiobook.mp3"
    combined.export(output_path, format="mp3")

    for i in range(len(chunks)):
        os.remove(f"outputs/{job_id}_chunk_{i}.wav")

    return {"audio_url": f"/audio/{job_id}_audiobook.mp3"}

@app.get("/audio/{filename}")
def get_audio(filename: str):
    path = f"outputs/{filename}"
    if not os.path.exists(path):
        raise HTTPException(status_code=404, detail="File not found")
    return FileResponse(path, media_type="audio/mpeg")
'''

with open('main.py', 'w') as f:
    f.write(app_code)

print('main.py written.')

In [ ]:
# Step 5: Set your ngrok auth token (get free token at https://ngrok.com)
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # <-- replace this

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)

In [ ]:
# Step 6: Start FastAPI + expose via ngrok
import threading
import uvicorn
import sys
sys.path.insert(0, '.')

from main import app

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()

public_url = ngrok.connect(8000)
print(f"\n✅ Backend is live at: {public_url}")
print(f"\n👉 Copy this URL and paste it in your Next.js app instead of http://localhost:8000")